# Numeric Stability Memo

## Cross Entropy

At each forward-pass step, the transformer language model outputs a single `logits` tensor at step t of size `(batch, vocab)`, where `logits[b][v]` denotes b-th sample's vocab v's score.

One reasonable probabilistic model to fit for this `logits` is `softmax`. It cross-checks a `targets` tensor of shape `(batch,)`, whose b-th element indicts the target vocab id for b-th sample in the batch. Then `logits[b, targets[b]]` gives a score ranging from -inf to inf. And finally, we can apply `softmax` to turn it into a probability:

$$
\frac{\exp(\text{logits}[b, \text{targets}[b]])}{\sum_{v} \exp(\text{logits}[b, v])}
$$

Name of game now is to maximize this probability with respect to transfomer's weights, which are the inputs to `logits`. This is a simple backpropagation:
1. Given `logits` and `targets`, compute `softmax` as above.
2. Backprop the computation graph that produced `logits`, to compute gradient.
3. Perform gradient *ascent* to maximize the probability.

This unfortunately doesn't work as-is, due to numeric stabiility where `exp(logit)` can be arbitrarily large. A common trick is to apply `log`, and by convention we negate it to perform gradient *descent*:

$$
-\log\left(\frac{\exp(\text{logits}[b, \text{target}[b]])}{\sum_{v} \exp(\text{logits}[b, v])}\right)
$$

This is the classic **cross-entropy loss**. Unfortunately the exponentials are still risky to hit inf. The numerator is trivial to address as `log(exp(x)) == x`, but the denominator has a log of sum of exponents.

Enters "log sum exp" trick. When calculating softmax, we can actually multiply numerator and denominator by the same `exp(-C)`, where `C` can be any constant. So suppose for each batch sample b, we let `C` be the max over all logits. Then:
* Numerator is just an exponent, within a reasonable range of original logits values.
* Each term in the sum in denominator can be at most 1 (`e^0 == 1`), and at most 0 (`e^-inf == 0`). When you sum over all, the total is at least 1, and at most `V` (since there are exactly `V` terms in the sum). Taking `log` over the denominator is then numerically stable, as the argument inside log is bounded.